## Черновик генератора просто чекнуть насколько адекватные вообще шаблоны генерируются
P.S. Извиняюсь за не очень полит. корректное название файла :)

In [6]:
from dataclasses import dataclass
from typing import List, Dict, Any
from datetime import datetime, timedelta
import random

from examples import Dates, Times, Locations, People, Events, Urls
from templates import MessageTemplate, TEMPLATES


dates_ex = Dates()
times_ex = Times()
locations_ex = Locations()
people_ex = People()
events_ex = Events()
urls_ex = Urls()


def sample_date(lang: str = "ru") -> str:
    """Сгенерировать дату в одном из заданных форматов."""
    base = datetime(2025, 12, 6)
    d = base + timedelta(days=random.randint(0, 30))

    if lang == "ru":
        formats = dates_ex.full_date_formats + dates_ex.no_year_formats
    else:
        formats = dates_ex.full_date_formats

    fmt = random.choice(formats)
    return d.strftime(fmt)


def sample_time(lang: str = "ru") -> str:
    """Сгенерировать время в одном из заданных форматов."""
    h = random.randint(8, 20)
    m = random.choice([0, 15, 30, 45])
    dt = datetime(2025, 1, 1, h, m)

    if lang == "ru":
        formats = times_ex.digital_24h_formats
    else:
        formats = times_ex.digital_12h_formats + times_ex.digital_24h_formats

    fmt = random.choice(formats)
    return dt.strftime(fmt)


def sample_place(lang: str = "ru") -> str:
    """Сгенерировать место: комната или онлайн-платформа."""
    candidates: List[str] = []

    if lang == "ru":
        if hasattr(locations_ex, "rooms_ru"):
            candidates += locations_ex.rooms_ru
        if hasattr(locations_ex, "online_ru"):
            candidates += locations_ex.online_ru
    else:
        if hasattr(locations_ex, "rooms_en"):
            candidates += locations_ex.rooms_en
        if hasattr(locations_ex, "online_en"):
            candidates += locations_ex.online_en

    return random.choice(candidates) if candidates else "office"


def sample_people(lang: str = "ru") -> str:
    """Сгенерировать людей: группа, роль или имя."""
    candidates: List[str] = []

    if lang == "ru":
        if hasattr(people_ex, "groups_ru"):
            candidates += people_ex.groups_ru
        if hasattr(people_ex, "roles_ru"):
            candidates += people_ex.roles_ru
        if hasattr(people_ex, "names_ru"):
            candidates += people_ex.names_ru
    else:
        if hasattr(people_ex, "groups_en"):
            candidates += people_ex.groups_en
        if hasattr(people_ex, "roles_en"):
            candidates += people_ex.roles_en
        if hasattr(people_ex, "names_en"):
            candidates += people_ex.names_en

    return random.choice(candidates) if candidates else "team"


def sample_event_name(lang: str = "ru") -> str:
    """Сгенерировать название события."""
    if lang == "ru":
        type_ = random.choice(events_ex.types_ru)
        proj = random.choice(events_ex.project_names)
        return f"{type_} {proj}"
    else:
        type_ = random.choice(events_ex.types_en)
        topic = random.choice(events_ex.topics_en)
        proj = random.choice(events_ex.project_names)
        pattern = random.choice([
            f"{type_} about {proj}",
            f"{proj} {topic}",
            f"{type_} {topic}",
        ])
        return pattern


def sample_link() -> str:
    """Сгенерировать ссылку на звонок по одному из шаблонов Urls."""
    key, tmpl = random.choice(list(urls_ex.templates.items()))

    if "{id}" in tmpl:
        return tmpl.format(id=random.randint(10**8, 10**9 - 1))

    if "{code}" in tmpl:
        letters = "abcdefghijklmnopqrstuvwxyz"
        code = (
            "".join(random.choice(letters) for _ in range(3))
            + "-"
            + "".join(random.choice(letters) for _ in range(4))
        )
        return tmpl.format(code=code)

    if "{uid}" in tmpl:
        uid = "19:" + "".join(random.choice("0123456789abcdef") for _ in range(30))
        return tmpl.format(uid=uid)

    return tmpl


def sample_slot(slot: str, lang: str = "ru") -> str:
    """Вернуть одно случайное значение для данного слота."""
    if slot == "date":
        return sample_date(lang)
    if slot == "time":
        return sample_time(lang)
    if slot == "place":
        return sample_place(lang)
    if slot == "people":
        return sample_people(lang)
    if slot == "event_name":
        return sample_event_name(lang)
    if slot == "link":
        return sample_link()
        
    return slot


def generate_message(template: MessageTemplate, lang: str = "ru") -> Dict[str, Any]:
    """
    Заполнить один шаблон конкретными значениями.
    Возвращает словарь с текстом и фактическими значениями слотов.
    """
    values: Dict[str, str] = {}

    for slot in template.slots:
        values[slot] = sample_slot(slot, lang)

    text_template = template.ru if lang == "ru" else template.en
    text = text_template.format(**values)

    return {
        "text": text,
        "lang": lang,
        "slots": values,
    }


def generate_dataset(n: int, lang: str = "ru") -> List[Dict[str, Any]]:
    """
    Сгенерировать n примеров для заданного языка.
    """
    return [
        generate_message(random.choice(TEMPLATES), lang)
        for _ in range(n)
    ]


In [7]:
ru_version = generate_dataset(5, 'ru')
for ex in ru_version:
    print(ex['text'], '\n')

en_version = generate_dataset(5, 'en')
for ex in en_version:
    print(ex['text'], '\n')

Забронируй переговорку Тимс под дейлик Alpha 17 12 2025 в 19:45, позови стажеры. 

Добавь в календарь дейлик MVP 2025-12-20 в 12.00. Без ссылки, просто отметь. 

Отмени, пожалуйста, интервью 'Ромашка', оно было на 02-01-2026 в 13.30. 

Нужно созвониться с менеджер. Поставь звонок 12/12/2025 на 18-00 в https://zoom.us/j/803844477. 

Если можно, сдвинь встреча Project X на попозже — на 12-30 09 12, место оставь Discord. 

ASAP schedule a call on zoom.us/j/851081618 on 09.12.2025 around 12:15, I want to discuss this with John. 

Set up a meeting weekly about Alpha on 04 01 2026, around 18:00, at Auditorium, and invite client. 

Please create a meeting sync planning on 01.01.2026 at 19:00 and add John. 

Let's do 'Ромашка' status update on 05 01 2026 at 14.00, same place as usual — Google Meet. 

Serious zoom 19 12 2025, remind about it at 17-30. 



Немного лажает кажется (либо у меня шиза), можно как вариант попробовать указывать некоторые правила генерации по типу нельзя одновременно использовать в одном шаблоне тип мероприятия "1-1" и людей "all", а то ерунда с логической точки зрения получится. Хз насколько это критично правда, нужно ваше мнение, коллеги